# 第七週：文件分類（PTT 伊朗議題）

本 notebook 依照 `week7_classification_zh_udn.ipynb` 的分析流程，使用 `groupwork2` 的 PTT 伊朗議題資料完成文件分類作業。

## 分析目標
- 資料來源：PTT `Gossiping` 與 `Military` 兩個版別
- 研究主題：伊朗相關議題文章
- 分類任務：根據文章文字內容，預測文章屬於哪一個版別
- 對應範例：沿用範例 notebook 的「前處理 → DTM/TF-IDF → 分類模型 → 評估 → 模型比較 → 係數解釋 → 新資料預測」流程


## 0. 先整理範例流程與 groupwork2 可沿用資訊

### 範例 notebook 的主要步驟
1. 讀入資料並檢查版別分布與日期範圍
2. 清理文字、合併標題與內文、建立 `content`
3. 使用 jieba 斷詞並移除停用字，建立 `words`
4. 以 7:3 切分 train/test，建立 `CountVectorizer` 與 `TfidfVectorizer`
5. 先示範 Logistic Regression 的基本分類流程，再用 `classification_report` 與混淆矩陣評估
6. 使用 cross-validation 檢查模型穩定度
7. 比較多個分類器，選出表現最佳的模型
8. 用 Logistic Regression 係數觀察可解釋特徵
9. 將訓練好的分類器套用到另一份資料做預測

### groupwork2 目前可直接沿用的資訊
1. 主要資料檔：`../data/ptt_iran_20260301_0320_new.csv` 與 `../data/ptt_iran_20260301_0410_full.csv`
2. 既有命名：沿用 `artTitle`、`artContent`、`artCatagory`、`artUrl` 等欄位名稱
3. 既有辭典資源：`../dict/dict.txt`、`../dict/user_dict.txt`、`../dict/stopwords.txt`、`../dict/text_stopwords.txt`
4. week6 / Main notebook 已經整理過的 PTT 清理邏輯：網址、引述、簽名檔、方括號標籤與同義詞處理
5. `warptt.py` 顯示此專案資料來自 `Military` 與 `Gossiping` 兩版的伊朗關鍵字爬蟲

### 本 notebook 的對應調整
1. `ptt_iran_20260301_0320_new.csv` 原始檔有缺少 `artCatagory`、`artTitle`、`artContent` 的資料，也有重複 `artUrl`，因此在建模前會先清理
2. 為了貼近範例 notebook 的「訓練資料 + 新資料預測」設計，本 notebook 使用清理後的 `0320_new` 作為訓練資料，再從 `0410_full` 中排除掉訓練期重複文章，保留 2026-03-21 到 2026-04-10 的後續文章當作新資料集
3. 由於 `Gossiping` 與 `Military` 的比例明顯不平衡，因此 train/test 切分時會加入 `stratify=y`，這是依資料特性做的必要調整


## 1. 套件說明
下面列出本次 notebook 會用到的核心套件，對應範例 notebook 的第 1 節。

- `pandas`：整理表格資料與摘要統計
- `jieba`：中文斷詞
- `sklearn`：向量化、分類模型、cross-validation、模型評估
- `matplotlib` / `seaborn`：資料視覺化與混淆矩陣繪圖


In [1]:
import re
from pathlib import Path
from pprint import pprint

import jieba
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib import font_manager as fm
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_predict, cross_validate, train_test_split
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 777
BASE_DIR = Path.cwd()
DICT_DIR = BASE_DIR.parent / 'dict'
DATA_DIR = BASE_DIR.parent / 'data'

pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')


設定中文字體（對應範例 notebook 的中文字體設定）。這裡優先沿用 groupwork2 內附的 `font.ttf`。

In [2]:
font_path = BASE_DIR / 'font.ttf'
if font_path.exists():
    fm.fontManager.addfont(str(font_path))
    font_name = fm.FontProperties(fname=str(font_path)).get_name()
else:
    font_name = 'Arial Unicode MS'

plt.rcParams['font.sans-serif'] = [font_name]
plt.rcParams['axes.unicode_minus'] = False
print(f'using font: {font_name}')


using font: Noto Sans TC


## 2. 文字前處理
本節對應範例 notebook 的第 2 節，先讀入原始資料並檢查資料狀況，再完成清理、合併文字與斷詞。

- 訓練資料來源：`ptt_iran_20260301_0320_new.csv`
- 完整資料來源：`ptt_iran_20260301_0410_full.csv`


In [3]:
train_raw = pd.read_csv(DATA_DIR / 'ptt_iran_20260301_0320_new.csv')
full_raw = pd.read_csv(DATA_DIR / 'ptt_iran_20260301_0410_full.csv')


def summarize_raw_dataset(df, dataset_name):
    art_date = pd.to_datetime(df['artDate'], errors='coerce')
    return {
        'dataset': dataset_name,
        'rows': len(df),
        'unique_urls': df['artUrl'].nunique(dropna=True),
        'duplicate_urls': df['artUrl'].duplicated().sum(),
        'missing_label': df['artCatagory'].isna().sum(),
        'missing_title': df['artTitle'].isna().sum(),
        'missing_content': df['artContent'].isna().sum(),
        'date_min': art_date.min(),
        'date_max': art_date.max(),
    }

raw_summary = pd.DataFrame([
    summarize_raw_dataset(train_raw, 'ptt_iran_20260301_0320_new.csv'),
    summarize_raw_dataset(full_raw, 'ptt_iran_20260301_0410_full.csv'),
])

display(raw_summary)


,dataset,rows,unique_urls,duplicate_urls,missing_label,missing_title,missing_content,date_min,date_max
0,ptt_iran_20260301_0320_new.csv,2306,2117,188,230,193,230,2026-03-01 00:01:00,2026-03-23 23:28:00
1,ptt_iran_20260301_0410_full.csv,3547,3547,0,0,0,0,2026-03-01 00:01:58,2026-04-10 22:02:53


從上表可以看到，`0320_new` 原始檔和範例的新聞資料不同，存在缺值與重複文章；`0410_full` 則是較完整的版本。接下來會先把這些差異在前處理階段處理掉，再進入和範例一致的分類流程。

In [4]:
train_raw.head(3)


,system_id,artUrl,artTitle,artDate,artPoster,artCatagory,artContent,artComment,insertedDate,dataSource
0,1,https://www.ptt.cc/bbs/Military/M.1774018954.A.FA4.html,Re: [情報] 美軍一架F-35在伊朗執行任務臨時迫降,2026/3/20 23:02,jimmy5680,Military,https://x.com/i/status/2034972525222838351\n\nNPR報導\n\n迫降的F-35飛行員有破片輕傷\n受損的F-35進行硬著陸，短期內不能作戰\n\n\n另外網路上流傳據稱是伊朗防空系統的影...,"[{""cmtStatus"": ""推"", ""cmtPoster"": ""Hfy0920"", ""cmtContent"": "": 飛太低被抓到？"", ""cmtDate"": ""2026-03-20 23:04:00""}, {""cmtStatu...",2026/3/23 23:25,ptt
1,2,https://www.ptt.cc/bbs/Military/M.1774003270.A.AAD.html,[新聞] 川普考慮奪取Kharg Island 迫使伊朗開放,2026/3/20 18:41,chordate,Military,原文來源：\nhttps://www.axios.com/2026/03/20/iran-invasion-kharg-island-strait-hormuz\n\n原文摘要（Gemini機翻）：\n\n川普考慮採取冒險的Khar...,"[{""cmtStatus"": ""噓"", ""cmtPoster"": ""a8785007"", ""cmtContent"": "": TACO"", ""cmtDate"": ""2026-03-20 18:46:00""}, {""cmtStatus""...",2026/3/23 23:25,ptt
2,3,https://www.ptt.cc/bbs/Military/M.1773998257.A.4B3.html,Re: [分享] 川普以珍珠港事件來形容打伊朗的出其不意,2026/3/20 17:17,scarfman,Military,※ 引述《andyken (碎夢殘刀)》之銘言：\n: https://x.com/JournalistJet/status/2034665847189393703\n: https://x.com/Acyn/status/2034...,"[{""cmtStatus"": ""推"", ""cmtPoster"": ""greedypeople"", ""cmtContent"": "": 那個記者看起來就是幫中國去亂的"", ""cmtDate"": ""2026-03-20 17:20:00""...",2026/3/23 23:25,ptt


In [5]:
label_preview = pd.concat(
    {
        'train_raw': train_raw['artCatagory'].value_counts(dropna=False),
        'full_raw': full_raw['artCatagory'].value_counts(dropna=False),
    },
    axis=1,
).fillna(0).astype(int)

label_preview


,train_raw,full_raw
artCatagory,,
Gossiping,1546,2616
Military,530,931
NaN,230,0


### 2.1 清理
這一步對應範例 notebook 的 2.1 清理，但會沿用 groupwork2 在 week6 / Main notebook 已出現的 PTT 專屬清理邏輯。

處理重點：
- 移除缺少標題、內文或版別的資料
- 移除重複 `artUrl`
- 清掉網址、PTT 發信站資訊、引述、回文標籤與雜訊符號
- 合併 `artTitle` 與 `artContent` 成新的 `content`


In [6]:
custom_words = [
    '防衛隊', '革命衛隊', '無人機', '核設施', '哈瑪斯', '真主黨', '中東',
    '大馬士革', '德黑蘭', '哈梅內伊', '哈米尼', '霍爾木茲海峽',
    '荷姆茲海峽', '荷莫茲海峽'
]

synonyms = {
    '哈梅內伊': '哈米尼',
    '霍爾木茲': '荷姆茲',
    '荷莫茲': '荷姆茲',
    '阿川': '川普',
    '川皇': '川普',
}

manual_stopwords = {
    '-----', '----', '連結', '嘖嘖', '醒醒', '裝睡', '衛生紙', '乳摸', '高潮',
    '新聞', '內文', '備註', '刪除', '完整', '轉載', '編輯', '張貼', '署名',
    '文章', '標題', '報導', '八卦', '看板', '作者', '時間', '來源', '內容',
    '媒體', '情況', '問題', '如題', '原因', '網址', '發信', '實業', 'cc',
    'ptt', 'gossiping', 'www', 'com', 'https', 'http', 'share', 'url', 'index',
    'html', 'bbs', 'status', '表示', '行動', '國家', '當作', '出現', '晚間',
    '政府', '目前', '請放', '最後', '違者', '未滿', '繁體', '中文字', '水桶',
    '一天', '以天', '單位', '網友', '點擊', '情報', '分享', '問卦', '討論',
    '心得', '爆卦'
}

jieba.set_dictionary(str(DICT_DIR / 'dict.txt'))
jieba.load_userdict(str(DICT_DIR / 'user_dict.txt'))
for word in custom_words:
    jieba.add_word(word)

stopwords = []
for stopword_file in [DICT_DIR / 'stopwords.txt', DICT_DIR / 'text_stopwords.txt']:
    with open(stopword_file, encoding='utf-8') as f:
        stopwords.extend([line.strip() for line in f if line.strip()])
stopwords = set(stopwords) | manual_stopwords


def clean_ptt_text(text):
    text = str(text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'※\s*(發信站|文章網址|編輯).*', '', text)
    text = re.sub(r'引述《.*?》之銘言', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'^[>:]\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'[a-zA-Z0-9]+', '', text)
    text = re.sub(r'[-/=_~]{2,}', '', text)
    text = re.sub(r'[^\u4e00-\u9fa5\n]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def getToken(text):
    tokens = []
    for word in jieba.lcut(text):
        word = synonyms.get(word, word).strip()
        if len(word) >= 2 and word not in stopwords:
            tokens.append(word)
    return tokens


def prepare_ptt_documents(df, drop_missing_label=True):
    data = df[['artUrl', 'artTitle', 'artContent', 'artCatagory', 'artDate']].copy()
    subset = ['artTitle', 'artContent'] + (['artCatagory'] if drop_missing_label else [])
    data = data.dropna(subset=subset)
    data = data.drop_duplicates(subset='artUrl').copy()
    data['artDate'] = pd.to_datetime(data['artDate'], errors='coerce')
    data['title_clean'] = data['artTitle'].map(clean_ptt_text)
    data['content_clean'] = data['artContent'].map(clean_ptt_text)
    data['content'] = (data['title_clean'] + ' ' + data['content_clean']).str.strip()
    data['words'] = data['content'].apply(getToken).map(' '.join)
    data = data[data['words'].str.len() > 0].copy()
    return data


def plot_confusion(y_true, y_pred, classes, title):
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap=plt.cm.Blues, cbar=False, ax=ax)
    ax.set(
        xlabel='Pred',
        ylabel='True',
        xticklabels=classes,
        yticklabels=classes,
        title=title,
    )
    plt.yticks(rotation=0)
    plt.show()
    return cm


Building prefix dict from /Users/wuchunting/Downloads/SMA_2026S-main/Social_Network_Analysis_Group_3_work2-main/groupwork2/dict/dict.txt ...


Loading model from cache /var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/jieba.u3cdd2ac410c7ea92c26c4a2c6e3f75b6.cache


Loading model cost 0.165 seconds.


Prefix dict has been built successfully.


文章的標題 `artTitle` 和內文 `artContent` 都會納入分析內容，並保留 `artUrl` 與 `artCatagory` 供後續建模與比對使用。

In [7]:
train_docs = prepare_ptt_documents(train_raw, drop_missing_label=True)
full_docs = prepare_ptt_documents(full_raw, drop_missing_label=True)

later_docs = full_docs[
    (~full_docs['artUrl'].isin(set(train_docs['artUrl'])))
    & (full_docs['artDate'] > train_docs['artDate'].max())
].copy()

clean_summary = pd.DataFrame([
    {
        'dataset': 'train_docs',
        'rows_after_clean': len(train_docs),
        'date_min': train_docs['artDate'].min(),
        'date_max': train_docs['artDate'].max(),
        'Gossiping': (train_docs['artCatagory'] == 'Gossiping').sum(),
        'Military': (train_docs['artCatagory'] == 'Military').sum(),
    },
    {
        'dataset': 'full_docs',
        'rows_after_clean': len(full_docs),
        'date_min': full_docs['artDate'].min(),
        'date_max': full_docs['artDate'].max(),
        'Gossiping': (full_docs['artCatagory'] == 'Gossiping').sum(),
        'Military': (full_docs['artCatagory'] == 'Military').sum(),
    },
    {
        'dataset': 'later_docs_for_section6',
        'rows_after_clean': len(later_docs),
        'date_min': later_docs['artDate'].min(),
        'date_max': later_docs['artDate'].max(),
        'Gossiping': (later_docs['artCatagory'] == 'Gossiping').sum(),
        'Military': (later_docs['artCatagory'] == 'Military').sum(),
    },
])

display(clean_summary)


,dataset,rows_after_clean,date_min,date_max,Gossiping,Military
0,train_docs,2075,2026-03-01 00:01:00,2026-03-20 23:45:00,1545,530
1,full_docs,3547,2026-03-01 00:01:58,2026-04-10 22:02:53,2616,931
2,later_docs_for_section6,1464,2026-03-21 00:18:35,2026-04-10 22:02:53,1065,399


In [8]:
train_docs[['artDate', 'artCatagory', 'artTitle', 'content', 'words']].head(3)


,artDate,artCatagory,artTitle,content,words
0,2026-03-20 23:02:00,Military,Re: [情報] 美軍一架F-35在伊朗執行任務臨時迫降,美軍一架 在伊朗執行任務臨時迫降 報導 迫降的 飛行員有破片輕傷 受損的 進行硬著陸 短期內不能作戰 另外網路上流傳據稱是伊朗防空系統的影片 我不知道真假 不過看起來應該是紅外線,美軍 一架 伊朗 執行 任務 臨時 迫降 迫降 飛行員 破片 輕傷 受損 進行 硬著陸 短期 作戰 網路 流傳 伊朗 防空 系統 影片 知道 真假 紅外線
1,2026-03-20 18:41:00,Military,[新聞] 川普考慮奪取Kharg Island 迫使伊朗開放,川普考慮奪取 迫使伊朗開放 原文來源 原文摘要 機翻 川普考慮採取冒險的 奪取行動 以迫使伊朗開放海峽 四名知情人士告訴 川普政府正考慮占領或封鎖伊朗的哈爾克島 以施壓 伊朗重新開放霍姆茲海峽 但占領哈爾克島的行動可能會讓美國軍隊更...,川普 考慮 奪取 迫使 伊朗 開放 原文 原文 摘要 機翻 川普 考慮 採取 冒險 奪取 迫使 伊朗 開放 海峽 四名 知情 人士 告訴 川普 考慮 占領 封鎖 伊朗 哈爾 克島 施壓 伊朗 重新 開放 霍姆茲 海峽 占領 哈爾 克...
2,2026-03-20 17:17:00,Military,Re: [分享] 川普以珍珠港事件來形容打伊朗的出其不意,川普以珍珠港事件來形容打伊朗的出其不意 日本記者 在您打算攻擊伊朗之前 為什麼沒有事先通知您的盟友 川普 我們希望這是一個驚喜 還有誰比日本更了解 驚喜 那你們當年為什麼沒有告 訴我們會去打珍珠港 旁邊高市聽到珍珠港那段的表情感覺很...,川普 珍珠港 事件 形容 伊朗 出其不意 日本 記者 打算 攻擊 伊朗 事先 通知 盟友 川普 希望 驚喜 日本 了解 驚喜 當年 珍珠港 旁邊 高市 聽到 珍珠港 那段 表情 感覺 微妙 知道 川普 強調 出其不意 認真 說起來 ...


### 2.2 斷詞
對應範例 notebook 的 2.2 節。這裡使用 groupwork2 的專案字典、使用者字典、停用字與 week6 中已整理過的同義詞規則。

In [9]:
train_docs[['artCatagory', 'words']].head(5)


,artCatagory,words
0,Military,美軍 一架 伊朗 執行 任務 臨時 迫降 迫降 飛行員 破片 輕傷 受損 進行 硬著陸 短期 作戰 網路 流傳 伊朗 防空 系統 影片 知道 真假 紅外線
1,Military,川普 考慮 奪取 迫使 伊朗 開放 原文 原文 摘要 機翻 川普 考慮 採取 冒險 奪取 迫使 伊朗 開放 海峽 四名 知情 人士 告訴 川普 考慮 占領 封鎖 伊朗 哈爾 克島 施壓 伊朗 重新 開放 霍姆茲 海峽 占領 哈爾 克...
2,Military,川普 珍珠港 事件 形容 伊朗 出其不意 日本 記者 打算 攻擊 伊朗 事先 通知 盟友 川普 希望 驚喜 日本 了解 驚喜 當年 珍珠港 旁邊 高市 聽到 珍珠港 那段 表情 感覺 微妙 知道 川普 強調 出其不意 認真 說起來 ...
3,Military,訪問 伊朗 德黑蘭 教授 影片 時有 觀看 建議 觀看 得到 美國 訊息 影片 教授 明確 指出 伊朗 打擊 阿聯 沙烏地 油氣 基礎 設施 美以 聯軍 空襲 暗殺 直接 報復 頻道 看到 美國 官方 觀點 翻譯 中文 頻道 翻譯 ...
5,Military,伊朗 襲擊 約旦河 西岸 巴勒斯坦人 喪命 原文 原文 摘要 中央社 耶路撒冷 綜合 外電 巴勒斯坦 新月 伊朗 昨天 深夜 占領 約旦河 西岸 發動 飛彈 攻擊 造成 巴勒斯坦 婦女 喪命 美以 伊朗 開戰 伊朗 首度 約旦河 西...


### 2.3 資料集基本檢視
對應範例 notebook 的 2.3 節，先確認清理後可用於分類的文章數與版別分布。

In [10]:
print(f"total posts: {len(train_docs['artUrl'].unique())}")
print(f"date range: {train_docs['artDate'].min()} ~ {train_docs['artDate'].max()}")
print('category count:')
print(train_docs['artCatagory'].value_counts())

fig, ax = plt.subplots(figsize=(5, 4))
train_docs['artCatagory'].value_counts().plot(kind='bar', color=['#4C72B0', '#DD8452'], ax=ax)
ax.set_title('Cleaned Training Data Label Distribution')
ax.set_xlabel('artCatagory')
ax.set_ylabel('count')
plt.xticks(rotation=0)
plt.show()


total posts: 2075
date range: 2026-03-01 00:01:00 ~ 2026-03-20 23:45:00
category count:
artCatagory
Gossiping    1545
Military      530
Name: count, dtype: int64


/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3119/651778717.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. 分類模型的訓練流程
這一節對應範例 notebook 的第 3 節，先以 Logistic Regression 示範完整的分類流程，再比較不同文字表示方式與評估方法。

In [11]:
data = train_docs.copy()
X = data['words']
y = data['artCatagory']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(X_train.head())
print(y_train.head())


1186                                      光明 牌卡 預言 伊朗 倒台 餓死 抬頭 光明 譽為 世界 影子 出過 著名 牌卡 五角大廈 武漢 肺炎 小魯 才疏學淺 問各位 新百 萬大 光明 牌卡 預言 伊朗 倒台
1842                                                                 代替 伊朗 世界杯 伊朗 百分 之百 美國 主辦 世界杯 足總 有沒有 幾會 中國 中華隊 一場 附加賽 取代 伊朗
660     世界盃 伊朗 踏足 美國 賽場 國際 足總 世界盃 伊朗 踏足 美國 賽場 國際 足總 主席 強硬 表態 賽程 不變 麗台 運動報 健全 國際 足總 主席 因凡蒂諾 明確 表態 否決 伊朗 將年 小組賽 美國 移師 墨西哥 請求 強...
954     快訊 伊朗 飛彈 猛攻 沙烏地 美軍 加油機 台灣 機會 川普 期中 選舉 願意 地面 部隊 以色列 黎巴嫩 南部 真主黨 騰不 出手 伊朗 現在 台灣 表現 越戰 時期 美國 民主 陣營 募集 志願軍 派往 越南 戰場 南韓 積極...
1183                                                                                                伊朗 二次 記得 前幾天 公開 露面 這一 兩天 發生
Name: words, dtype: object
1186    Gossiping
1842    Gossiping
660     Gossiping
954     Gossiping
1183    Gossiping
Name: artCatagory, dtype: object


In [12]:
print('raw data percentage:')
print((data['artCatagory'].value_counts(normalize=True) * 100).round(2))
print('\ntrain percentage:')
print((y_train.value_counts(normalize=True) * 100).round(2))
print('\ntest percentage:')
print((y_test.value_counts(normalize=True) * 100).round(2))


raw data percentage:
artCatagory
Gossiping    74.46
Military     25.54
Name: proportion, dtype: float64

train percentage:
artCatagory
Gossiping    74.45
Military     25.55
Name: proportion, dtype: float64

test percentage:
artCatagory
Gossiping    74.48
Military     25.52
Name: proportion, dtype: float64


### 3.2 將文章轉為 DTM
DTM（document-term matrix）對應範例 notebook 的 3.2 節。

- `CountVectorizer`：以詞頻表示文章
- `TfidfVectorizer`：以 TF-IDF 權重表示文章


### 3.3 套入正式資料集
先依範例 notebook 的順序，使用 `CountVectorizer + LogisticRegression` 示範完整流程。

In [13]:
count_vectorizer = CountVectorizer(max_features=1000)
print(count_vectorizer)


CountVectorizer(max_features=1000)


In [14]:
vec_train_count = count_vectorizer.fit_transform(X_train)
count_vocab = count_vectorizer.get_feature_names_out()
count_df = pd.DataFrame(vec_train_count.toarray(), columns=count_vocab)
count_df.iloc[:5, :20]


,一下,一份,一位,一名,一堆,一場,一枚,一架,一次,一段,一直,一種,一艘,一部分,一項,一點,三日,上升,上午,上漲
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [15]:
vec_test_count = count_vectorizer.transform(X_test)
print(vec_train_count.shape)
print(vec_test_count.shape)


(1452, 1000)
(623, 1000)


In [16]:
count_logistic = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
count_logistic.fit(vec_train_count, y_train)
count_logistic


LogisticRegression(max_iter=2000, random_state=777)

In [17]:
count_logistic.classes_


array(['Gossiping', 'Military'], dtype=object)

使用 train set 訓練後，再用 test set 查看模型的分類結果。

In [18]:
y_pred_count = count_logistic.predict(vec_test_count)
y_pred_proba_count = count_logistic.predict_proba(vec_test_count)
print(y_pred_count[:10])


['Gossiping' 'Gossiping' 'Gossiping' 'Military' 'Gossiping' 'Military'
 'Gossiping' 'Gossiping' 'Military' 'Gossiping']


觀察看看模型輸出的類別機率。

In [19]:
print(y_pred_proba_count.shape)
pd.Series(y_pred_proba_count[0], index=count_logistic.classes_)


(623, 2)


Gossiping    0.9945
Military     0.0055
dtype: float64

### 3.4 模型評估
使用 `classification_report` 與混淆矩陣檢查 `CountVectorizer + LogisticRegression` 的表現。

In [20]:
print(classification_report(y_test, y_pred_count))


              precision    recall  f1-score   support

   Gossiping       0.87      0.91      0.89       464
    Military       0.70      0.62      0.66       159

    accuracy                           0.83       623
   macro avg       0.79      0.76      0.77       623
weighted avg       0.83      0.83      0.83       623



混淆矩陣（Confusion Matrix）可以更直接觀察模型在哪一個版別上較容易分錯。

In [21]:
classes = list(count_logistic.classes_)
cm_count = plot_confusion(y_test, y_pred_count, classes, 'CountVectorizer + LogisticRegression')
cm_count


/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3119/4059253539.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


array([[422,  42],
       [ 61,  98]])

### 3.5 TF-IDF
接著改用 `TfidfVectorizer`，比較和純詞頻 DTM 的差異。

In [22]:
tfidf_vectorizer = TfidfVectorizer(max_features=1000)
vec_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
vec_test_tfidf = tfidf_vectorizer.transform(X_test)
tfidf_vocab = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(vec_train_tfidf.toarray(), columns=tfidf_vocab)
tfidf_df.iloc[:5, :20]


,一下,一份,一位,一名,一堆,一場,一枚,一架,一次,一段,一直,一種,一艘,一部分,一項,一點,三日,上升,上午,上漲
0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.603214,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.067309,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0695,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0


In [23]:
tfidf_logistic = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
tfidf_logistic.fit(vec_train_tfidf, y_train)
y_pred_tfidf = tfidf_logistic.predict(vec_test_tfidf)
y_pred_proba_tfidf = tfidf_logistic.predict_proba(vec_test_tfidf)

print(classification_report(y_test, y_pred_tfidf))


              precision    recall  f1-score   support

   Gossiping       0.83      0.98      0.90       464
    Military       0.87      0.43      0.58       159

    accuracy                           0.84       623
   macro avg       0.85      0.71      0.74       623
weighted avg       0.84      0.84      0.82       623



### 3.6 Cross Validation
對應範例 notebook 的 cross-validation 說明。這裡先用 `TfidfVectorizer + LogisticRegression` 檢查 5-fold CV 的整體穩定度。

In [24]:
cv_logistic = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
cv_train_matrix = TfidfVectorizer(max_features=1000).fit_transform(X_train)

cv_scores = cross_validate(
    cv_logistic,
    cv_train_matrix,
    y_train,
    cv=5,
    scoring=('accuracy', 'f1_weighted', 'precision_weighted', 'recall_weighted'),
    return_train_score=False,
)

cv_summary = pd.DataFrame(cv_scores).agg(['mean', 'std']).T
cv_summary


,mean,std
fit_time,0.004009,0.000353
score_time,0.002760,0.000223
test_accuracy,0.825761,0.010708
test_f1_weighted,0.801051,0.012299
test_precision_weighted,0.829680,0.018757
test_recall_weighted,0.825761,0.010708


`cross_val_predict()` 會回傳每一筆訓練資料在 cross-validation 下得到的預測類別。

In [25]:
y_pred_cv = cross_val_predict(cv_logistic, cv_train_matrix, y_train, cv=5)
print(classification_report(y_train, y_pred_cv))


              precision    recall  f1-score   support

   Gossiping       0.82      0.98      0.89      1081
    Military       0.84      0.39      0.53       371

    accuracy                           0.83      1452
   macro avg       0.83      0.68      0.71      1452
weighted avg       0.83      0.83      0.80      1452



## 4. 比較不同模型效果
這一節對應範例 notebook 的第 4 節，固定使用 `TfidfVectorizer(max_features=1000)`，比較 Logistic Regression、Decision Tree、SVM 與 Random Forest。

In [26]:
def train_cv_experiment(vectorizer, clf, X, y, model_name, cv=5):
    vec_X = vectorizer.fit_transform(X).toarray()
    y_pred_cv = cross_val_predict(clf, vec_X, y, cv=cv)
    cls_report = classification_report(y, y_pred_cv, output_dict=True)
    print(f'===== {model_name} =====')
    print(classification_report(y, y_pred_cv))
    plot_confusion(y, y_pred_cv, sorted(pd.Series(y).unique()), f'{model_name} CV Confusion Matrix')
    clf.fit(vec_X, y)
    return {
        'report': cls_report,
        'model': clf,
        'vectorizer': vectorizer,
        'cv_pred': y_pred_cv,
    }


In [27]:
model_set = {
    'clf_logistic': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'clf_dtree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'clf_svm': svm.SVC(probability=True, random_state=RANDOM_STATE),
    'clf_rf': RandomForestClassifier(random_state=RANDOM_STATE),
}

result_set = {}
for model_name, model in model_set.items():
    result_set[model_name] = train_cv_experiment(
        TfidfVectorizer(max_features=1000),
        model,
        X_train,
        y_train,
        model_name,
        cv=5,
    )


===== clf_logistic =====
              precision    recall  f1-score   support

   Gossiping       0.82      0.98      0.89      1081
    Military       0.84      0.39      0.53       371

    accuracy                           0.83      1452
   macro avg       0.83      0.68      0.71      1452
weighted avg       0.83      0.83      0.80      1452



/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3119/4059253539.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


===== clf_dtree =====
              precision    recall  f1-score   support

   Gossiping       0.85      0.85      0.85      1081
    Military       0.56      0.55      0.56       371

    accuracy                           0.78      1452
   macro avg       0.71      0.70      0.70      1452
weighted avg       0.77      0.78      0.78      1452



/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3119/4059253539.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


===== clf_svm =====
              precision    recall  f1-score   support

   Gossiping       0.84      0.98      0.91      1081
    Military       0.87      0.48      0.62       371

    accuracy                           0.85      1452
   macro avg       0.86      0.73      0.76      1452
weighted avg       0.85      0.85      0.83      1452



/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3119/4059253539.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


===== clf_rf =====
              precision    recall  f1-score   support

   Gossiping       0.85      0.97      0.90      1081
    Military       0.85      0.48      0.62       371

    accuracy                           0.85      1452
   macro avg       0.85      0.73      0.76      1452
weighted avg       0.85      0.85      0.83      1452



/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3119/4059253539.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


分別觀察各個分類模型在 weighted avg 指標上的整體表現，再決定最終模型。

In [28]:
model_summary = pd.DataFrame([
    {
        'model': model_name,
        'accuracy': result['report']['accuracy'],
        'precision_weighted': result['report']['weighted avg']['precision'],
        'recall_weighted': result['report']['weighted avg']['recall'],
        'f1_weighted': result['report']['weighted avg']['f1-score'],
        'f1_macro': result['report']['macro avg']['f1-score'],
    }
    for model_name, result in result_set.items()
]).sort_values(by='f1_weighted', ascending=False)

model_summary


,model,accuracy,precision_weighted,recall_weighted,f1_weighted,f1_macro
2,clf_svm,0.847796,0.850452,0.847796,0.831150,0.760381
3,clf_rf,0.845730,0.846067,0.845730,0.829840,0.759326
0,clf_logistic,0.825758,0.828442,0.825758,0.801172,0.713456
1,clf_dtree,0.776171,0.774800,0.776171,0.775466,0.703992


找出 `weighted avg f1-score` 表現最好的模型，作為最終分類器。

In [29]:
best_model_name = model_summary.iloc[0]['model']
best_result = result_set[best_model_name]

print(f'best model: {best_model_name}')
pprint(best_result['report'])


best model: clf_svm
{'Gossiping': {'f1-score': 0.9051094890510949,
               'precision': 0.844551282051282,
               'recall': 0.9750231267345051,
               'support': 1081.0},
 'Military': {'f1-score': 0.6156521739130435,
              'precision': 0.8676470588235294,
              'recall': 0.477088948787062,
              'support': 371.0},
 'accuracy': 0.8477961432506887,
 'macro avg': {'f1-score': 0.7603808314820693,
               'precision': 0.8560991704374057,
               'recall': 0.7260560377607835,
               'support': 1452.0},
 'weighted avg': {'f1-score': 0.8311503541225707,
                  'precision': 0.8504524757031442,
                  'recall': 0.8477961432506887,
                  'support': 1452.0}}


In [30]:
best_model = best_result['model']
best_vectorizer = best_result['vectorizer']

y_pred_best_test = best_model.predict(best_vectorizer.transform(X_test).toarray())
print(classification_report(y_test, y_pred_best_test))
plot_confusion(y_test, y_pred_best_test, sorted(y_test.unique()), f'{best_model_name} on Held-out Test Set')


              precision    recall  f1-score   support

   Gossiping       0.85      0.97      0.91       464
    Military       0.86      0.52      0.65       159

    accuracy                           0.86       623
   macro avg       0.86      0.74      0.78       623
weighted avg       0.86      0.86      0.84       623



/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3119/4059253539.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


array([[451,  13],
       [ 77,  82]])

## 5. 分析可解釋模型的結果
對應範例 notebook 的第 5 節。雖然第 4 節的最佳模型不一定是 Logistic Regression，但 Logistic Regression 的係數最容易解釋，因此仍沿用範例 notebook 的做法，用 `TfidfVectorizer + LogisticRegression` 觀察字詞特徵。

In [31]:
def plot_coef(logistic_reg_model, feature_names, top_n=10):
    if len(logistic_reg_model.classes_) == 2:
        coef_df = pd.DataFrame(
            {
                logistic_reg_model.classes_[0]: -logistic_reg_model.coef_[0],
                logistic_reg_model.classes_[1]: logistic_reg_model.coef_[0],
            },
            index=feature_names,
        )
    else:
        coef_df = pd.DataFrame(
            logistic_reg_model.coef_.T,
            columns=logistic_reg_model.classes_,
            index=feature_names,
        )

    display(coef_df.head())

    for label in coef_df.columns:
        select_words = (
            coef_df[[label]]
            .sort_values(by=label, ascending=False)
            .iloc[np.r_[0:top_n, -top_n:0]]
        )
        colors = np.where(select_words[label] >= 0, 'darkseagreen', 'rosybrown')

        fig, ax = plt.subplots(figsize=(8, top_n * 0.8))
        ax.barh(select_words.index, select_words[label], color=colors)
        ax.invert_yaxis()
        ax.set_title(f'Coeff increase/decrease odds ratio of 「{label}」 label the most', loc='left', size=14)
        ax.set_ylabel('word', size=12)
        ax.set_xlabel('odds ratio', size=12)
        plt.show()


In [32]:
plot_coef(tfidf_logistic, tfidf_vectorizer.get_feature_names_out(), top_n=10)


,Gossiping,Military
一下,0.778137,-0.778137
一份,-0.120897,0.120897
一位,0.028664,-0.028664
一名,-0.408784,0.408784
一堆,0.322601,-0.322601


/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3119/2074342872.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3119/2074342872.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


從係數圖可以看到，模型確實學到兩個版別不同的語言風格。

- `Military` 端通常更容易出現軍事情勢、地緣政治與戰況相關詞，例如空襲、伊拉克、官員、荷姆茲海峽等。
- `Gossiping` 端則更常出現 PTT 問卦或社會討論口吻，例如台灣、有沒有、現在、美國等。
- 這也說明本次分類不只是辨識「伊朗」主題，而是同時辨識兩個版在敘事方式與詞彙選擇上的差異。


## 6. 用訓練好的分類器來預測後續完整資料
範例 notebook 最後會把分類器套用到另一份資料。由於 groupwork2 沒有第二個不同來源的資料集，這裡改成使用 `ptt_iran_20260301_0410_full.csv` 中「不在訓練資料內、且日期晚於訓練期」的文章，模擬真正的新資料預測。

In [33]:
later_summary = pd.DataFrame({
    'rows': [len(later_docs)],
    'date_min': [later_docs['artDate'].min()],
    'date_max': [later_docs['artDate'].max()],
    'Gossiping': [(later_docs['artCatagory'] == 'Gossiping').sum()],
    'Military': [(later_docs['artCatagory'] == 'Military').sum()],
})

later_summary


,rows,date_min,date_max,Gossiping,Military
0,1464,2026-03-21 00:18:35,2026-04-10 22:02:53,1065,399


這份 `later_docs` 的時間範圍是 **2026-03-21 到 2026-04-10**，可視為訓練期之後的新文章。因為其版別欄位仍存在，所以我們除了做預測，也可以直接檢查實際表現。

In [34]:
X_new = later_docs['words']
y_new = later_docs['artCatagory']

y_pred_new = best_model.predict(best_vectorizer.transform(X_new).toarray())
print(classification_report(y_new, y_pred_new))


              precision    recall  f1-score   support

   Gossiping       0.84      0.95      0.89      1065
    Military       0.81      0.51      0.62       399

    accuracy                           0.83      1464
   macro avg       0.82      0.73      0.76      1464
weighted avg       0.83      0.83      0.82      1464



In [35]:
plot_confusion(y_new, y_pred_new, sorted(y_new.unique()), f'{best_model_name} on Later Unseen Posts')


/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3119/4059253539.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


array([[1017,   48],
       [ 197,  202]])

In [36]:
later_docs = later_docs.copy()
later_docs['pred'] = y_pred_new
later_docs[['artDate', 'artCatagory', 'pred', 'artTitle', 'words']].head(10)


,artDate,artCatagory,pred,artTitle,words
0,2026-04-10 20:12:41,Military,Gossiping,[新聞] 保護伊朗人員前來 巴基斯坦空軍闢安全,保護 伊朗 人員 前來 巴基斯坦 空軍 安全 出處 摘要 中央社 新德里 綜合 外電 美國 伊朗 將於日 巴基斯坦 首都 伊斯蘭 巴德 舉行 萬眾矚目 談判 外媒 披露 巴基斯坦 空軍 出動 大批 軍機 波斯灣 空域 保護 伊朗 代...
1,2026-04-10 16:20:52,Military,Gossiping,[情報] 幾則關於伊朗戰爭的民調,幾則 伊朗 戰爭 民調 統整 各間 民調 美國 民眾 伊朗 戰爭 支持度 持續 下滑 民調 追蹤 美國 民眾 以色列 納坦雅胡 支持度 顯示 美國 民眾 越來越 厭惡 以色列 納坦雅胡 天空 民調 顯示 去年 年底 現在 英國 民眾...
2,2026-04-10 15:54:15,Military,Gossiping,[分享] 大西洋期刊：中國從伊朗戰爭學到什麼,大西洋 期刊 中國 伊朗 戰爭 學到 大西洋 期刊 這篇 不予置評 參考 中國 伊朗 戰爭 中學 台灣 實施 封鎖 全球 經濟 造成 衝擊 將比 伊朗 封鎖 荷姆茲海峽 嚴重 中國 這場 伊朗 戰爭 中學 美國 劃下 紅線 期限 伴...
3,2026-04-10 15:42:14,Military,Military,[情報] 報導稱以色列軍方認為伊朗政府更為極端,以色列 軍方 認為 伊朗 以色列 以色列 軍方 告訴 國會 議員 伊朗 戰前 以色列 軍方 另稱 作戰 伊朗 造成 嚴重 打擊 伊朗 著手 恢復 飛彈 生產 能力 肖像 掛在 辦公室 掛上 孩子 照片 決定 看看 眼睛
4,2026-04-10 14:09:37,Military,Gossiping,[新聞] 曾嗆發展核武，伊朗前外長傷重不治,曾嗆 發展 核武 伊朗 外長 傷重 不治 揚言 發展 核武 伊朗 外長 拉齊 遇襲 住所 攻擊 罹難 伊朗 伊朗 外交部長 哈拉 美國 以色列 空襲 受傷 今天 傷重 不治 哈拉 德黑蘭 尋求 發展 核武 法新社 哈拉 濟生 隸屬於...
5,2026-04-10 10:29:51,Military,Gossiping,[情報] WSJ：中國想拿斡旋伊朗換川普反台獨,中國 斡旋 伊朗 川普 台獨 華爾街日報 中共 幫忙 斡旋 伊朗 停火 協議 希望 能夠 當做 討好 川普 籌碼 中共 希望 人情 中國 國民黨 主席 行程 宣傳 愛好 和平 遊說 川普 支持 台灣 反抗 承認 台灣 分離 省份 統一
6,2026-04-10 09:09:45,Military,Military,[情報] ISW 伊朗更新特別報告，2026年4月9日,伊朗 更新 特別 報告 伊朗 更新 特別 報告 戰爭 研究所 美國 企業 研究所 旗下 關鍵 威脅 計畫 正在 發布 每日 更新 提供 伊朗 戰爭 分析 更新 聚焦 美國 以色列 伊朗 打擊 伊朗 抵抗 軸心 打擊 回應 更新 涵蓋...
7,2026-04-10 07:18:37,Military,Military,[新聞] 美民主黨議員試圖限制川普對伊朗動武權,民主黨 議員 試圖 限制 川普 伊朗 動武 原文 原文 摘要 民主黨 議員 試圖 限制 川普 伊朗 動武 權限 共和黨 擋下 中央社 華盛頓 綜合 外電 美國 民主黨 聯邦 眾議員 限制 總統 川普 伊朗 動武 權限 今天 試圖 程...
8,2026-04-10 05:38:26,Military,Military,[情報] 南韓宣布要派特使去和伊朗談判,南韓 宣布 特使 伊朗 談判 南韓 外交部長 伊朗 外長 昨天 進行 通話 南韓 宣布 派遣 一名 特使 伊朗 進行 談判 停火 荷姆茲海峽 航運 議題 南韓 相關 船隻 困在 波灣 南韓 外交官員 優先 目標 確認 伊朗 海峽 通...
9,2026-04-10 05:10:56,Military,Military,Re: [情報] 川普對伊朗談判相關消息數則,川普 伊朗 談判 相關 消息 數則 泰晤士報 專家 荷姆茲海峽 航運 重點 軍艦 護航 護航 僅能 應付 運量 船東 害怕 美軍 護航 捲入 戰火 報稱 川普 談判 同意 停火 涵蓋 中東 包含 黎巴嫩 以色列 首肯 川普 納坦雅胡...


將錯誤分類的結果篩選出來，觀察哪些文章最容易混淆。

In [37]:
false_pred = later_docs.query('artCatagory != pred').loc[:, ['artDate', 'artCatagory', 'pred', 'artTitle', 'words']]
print(f'錯誤預測筆數: {len(false_pred)} / {len(later_docs)}')
false_pred.head(20)


錯誤預測筆數: 245 / 1464


,artDate,artCatagory,pred,artTitle,words
0,2026-04-10 20:12:41,Military,Gossiping,[新聞] 保護伊朗人員前來 巴基斯坦空軍闢安全,保護 伊朗 人員 前來 巴基斯坦 空軍 安全 出處 摘要 中央社 新德里 綜合 外電 美國 伊朗 將於日 巴基斯坦 首都 伊斯蘭 巴德 舉行 萬眾矚目 談判 外媒 披露 巴基斯坦 空軍 出動 大批 軍機 波斯灣 空域 保護 伊朗 代...
1,2026-04-10 16:20:52,Military,Gossiping,[情報] 幾則關於伊朗戰爭的民調,幾則 伊朗 戰爭 民調 統整 各間 民調 美國 民眾 伊朗 戰爭 支持度 持續 下滑 民調 追蹤 美國 民眾 以色列 納坦雅胡 支持度 顯示 美國 民眾 越來越 厭惡 以色列 納坦雅胡 天空 民調 顯示 去年 年底 現在 英國 民眾...
2,2026-04-10 15:54:15,Military,Gossiping,[分享] 大西洋期刊：中國從伊朗戰爭學到什麼,大西洋 期刊 中國 伊朗 戰爭 學到 大西洋 期刊 這篇 不予置評 參考 中國 伊朗 戰爭 中學 台灣 實施 封鎖 全球 經濟 造成 衝擊 將比 伊朗 封鎖 荷姆茲海峽 嚴重 中國 這場 伊朗 戰爭 中學 美國 劃下 紅線 期限 伴...
4,2026-04-10 14:09:37,Military,Gossiping,[新聞] 曾嗆發展核武，伊朗前外長傷重不治,曾嗆 發展 核武 伊朗 外長 傷重 不治 揚言 發展 核武 伊朗 外長 拉齊 遇襲 住所 攻擊 罹難 伊朗 伊朗 外交部長 哈拉 美國 以色列 空襲 受傷 今天 傷重 不治 哈拉 德黑蘭 尋求 發展 核武 法新社 哈拉 濟生 隸屬於...
5,2026-04-10 10:29:51,Military,Gossiping,[情報] WSJ：中國想拿斡旋伊朗換川普反台獨,中國 斡旋 伊朗 川普 台獨 華爾街日報 中共 幫忙 斡旋 伊朗 停火 協議 希望 能夠 當做 討好 川普 籌碼 中共 希望 人情 中國 國民黨 主席 行程 宣傳 愛好 和平 遊說 川普 支持 台灣 反抗 承認 台灣 分離 省份 統一
10,2026-04-10 01:57:20,Military,Gossiping,[情報] 伊朗副外長稱巴基斯坦說服己方別重啟戰端,伊朗 副外長 巴基斯坦 說服 己方 重啟 戰端 伊朗 外交部 副部長 受訪 聲稱 以色列 黎巴嫩 停止 攻擊 伊朗 願意 巴基斯坦 參加 談判 他稱 伊朗 昨天 準備 報復 以色列 巴基斯坦 高層 說服 避免 重啟 戰端 美國 施壓...
14,2026-04-09 15:24:26,Military,Gossiping,[新聞] 美伊談判生變？伊朗大使刪除伊朗代表團將,美伊 談判 生變 伊朗 大使 伊朗 代表團 美伊 談判 生變 伊朗 大使 伊朗 代表團 抵達 巴基斯坦 貼文 聯合報 大陸 中心 即時 美國 伊朗 原定 上午 巴基斯坦 首都 伊斯蘭 巴德 展開 首輪 談判 生變 新華社 發自 伊斯...
19,2026-04-09 09:50:21,Military,Gossiping,[情報] 小卡德羅夫:拒絕為伊朗作戰,卡德羅夫 拒絕 伊朗 作戰 車臣 領導人 卡德羅夫 支持 伊朗 該國 正在 攻擊 車臣 兄弟 順帶 一提 以色列 美國 回應 網路 流傳 說法 車臣 部隊 準備好 前往 伊朗 抵禦 發生 地面 軍事 卡德羅夫 補充 伊朗 對抗 美國...
21,2026-04-09 09:00:43,Military,Gossiping,Re: [情報] 伊朗的荷姆茲海峽收費機制,伊朗 荷姆茲海峽 收費 機制 戰敗國 囂張 川普 打死 川普 付錢 解決 好了 反正 川普 川普 確認 導彈 核武 達到 目的 伊朗 需要 重建 基金 做生意 吸毒 敗家子 加密 貨幣 利多 加密 貨幣 半年 突然 起漲 原來 川普...
28,2026-04-09 01:33:14,Military,Gossiping,Re: [情報] 川普稱雙方談判基礎並非伊朗提出的10點,川普 雙方 談判 基礎 伊朗 提出 的點 狐狸 美國 官員 先前 流傳 的點 條件 版本 官員 更新 方案 伊朗 部分 條件 妥協 描述 雙方 提到 方案 白宮 說明 方案 伊朗 公開 一份 宣傳 的點 方案 反正 死無對證 子彈 ...


分開觀察兩種常見錯誤：實際是 `Military` 卻被判成 `Gossiping`，以及實際是 `Gossiping` 卻被判成 `Military`。

In [38]:
false_pred.loc[false_pred['artCatagory'] == 'Military', :].head(10)


,artDate,artCatagory,pred,artTitle,words
0,2026-04-10 20:12:41,Military,Gossiping,[新聞] 保護伊朗人員前來 巴基斯坦空軍闢安全,保護 伊朗 人員 前來 巴基斯坦 空軍 安全 出處 摘要 中央社 新德里 綜合 外電 美國 伊朗 將於日 巴基斯坦 首都 伊斯蘭 巴德 舉行 萬眾矚目 談判 外媒 披露 巴基斯坦 空軍 出動 大批 軍機 波斯灣 空域 保護 伊朗 代...
1,2026-04-10 16:20:52,Military,Gossiping,[情報] 幾則關於伊朗戰爭的民調,幾則 伊朗 戰爭 民調 統整 各間 民調 美國 民眾 伊朗 戰爭 支持度 持續 下滑 民調 追蹤 美國 民眾 以色列 納坦雅胡 支持度 顯示 美國 民眾 越來越 厭惡 以色列 納坦雅胡 天空 民調 顯示 去年 年底 現在 英國 民眾...
2,2026-04-10 15:54:15,Military,Gossiping,[分享] 大西洋期刊：中國從伊朗戰爭學到什麼,大西洋 期刊 中國 伊朗 戰爭 學到 大西洋 期刊 這篇 不予置評 參考 中國 伊朗 戰爭 中學 台灣 實施 封鎖 全球 經濟 造成 衝擊 將比 伊朗 封鎖 荷姆茲海峽 嚴重 中國 這場 伊朗 戰爭 中學 美國 劃下 紅線 期限 伴...
4,2026-04-10 14:09:37,Military,Gossiping,[新聞] 曾嗆發展核武，伊朗前外長傷重不治,曾嗆 發展 核武 伊朗 外長 傷重 不治 揚言 發展 核武 伊朗 外長 拉齊 遇襲 住所 攻擊 罹難 伊朗 伊朗 外交部長 哈拉 美國 以色列 空襲 受傷 今天 傷重 不治 哈拉 德黑蘭 尋求 發展 核武 法新社 哈拉 濟生 隸屬於...
5,2026-04-10 10:29:51,Military,Gossiping,[情報] WSJ：中國想拿斡旋伊朗換川普反台獨,中國 斡旋 伊朗 川普 台獨 華爾街日報 中共 幫忙 斡旋 伊朗 停火 協議 希望 能夠 當做 討好 川普 籌碼 中共 希望 人情 中國 國民黨 主席 行程 宣傳 愛好 和平 遊說 川普 支持 台灣 反抗 承認 台灣 分離 省份 統一
10,2026-04-10 01:57:20,Military,Gossiping,[情報] 伊朗副外長稱巴基斯坦說服己方別重啟戰端,伊朗 副外長 巴基斯坦 說服 己方 重啟 戰端 伊朗 外交部 副部長 受訪 聲稱 以色列 黎巴嫩 停止 攻擊 伊朗 願意 巴基斯坦 參加 談判 他稱 伊朗 昨天 準備 報復 以色列 巴基斯坦 高層 說服 避免 重啟 戰端 美國 施壓...
14,2026-04-09 15:24:26,Military,Gossiping,[新聞] 美伊談判生變？伊朗大使刪除伊朗代表團將,美伊 談判 生變 伊朗 大使 伊朗 代表團 美伊 談判 生變 伊朗 大使 伊朗 代表團 抵達 巴基斯坦 貼文 聯合報 大陸 中心 即時 美國 伊朗 原定 上午 巴基斯坦 首都 伊斯蘭 巴德 展開 首輪 談判 生變 新華社 發自 伊斯...
19,2026-04-09 09:50:21,Military,Gossiping,[情報] 小卡德羅夫:拒絕為伊朗作戰,卡德羅夫 拒絕 伊朗 作戰 車臣 領導人 卡德羅夫 支持 伊朗 該國 正在 攻擊 車臣 兄弟 順帶 一提 以色列 美國 回應 網路 流傳 說法 車臣 部隊 準備好 前往 伊朗 抵禦 發生 地面 軍事 卡德羅夫 補充 伊朗 對抗 美國...
21,2026-04-09 09:00:43,Military,Gossiping,Re: [情報] 伊朗的荷姆茲海峽收費機制,伊朗 荷姆茲海峽 收費 機制 戰敗國 囂張 川普 打死 川普 付錢 解決 好了 反正 川普 川普 確認 導彈 核武 達到 目的 伊朗 需要 重建 基金 做生意 吸毒 敗家子 加密 貨幣 利多 加密 貨幣 半年 突然 起漲 原來 川普...
28,2026-04-09 01:33:14,Military,Gossiping,Re: [情報] 川普稱雙方談判基礎並非伊朗提出的10點,川普 雙方 談判 基礎 伊朗 提出 的點 狐狸 美國 官員 先前 流傳 的點 條件 版本 官員 更新 方案 伊朗 部分 條件 妥協 描述 雙方 提到 方案 白宮 說明 方案 伊朗 公開 一份 宣傳 的點 方案 反正 死無對證 子彈 ...


In [39]:
false_pred.loc[false_pred['artCatagory'] == 'Gossiping', :].head(10)


,artDate,artCatagory,pred,artTitle,words
972,2026-04-08 18:42:01,Gossiping,Military,[新聞] 國會未授權攻打伊朗 傳共和黨議員要求,國會 授權 攻打 伊朗 共和黨 議員 要求 民視 記者 徐子 國會 授權 攻打 伊朗 共和黨 議員 要求 川普 這天 停止 軍事 美國 伊朗 達成 兩週 暫時 停火 協議 中東 前景 依舊 未明 美國 共和黨 面臨 國會 期中 想要...
986,2026-04-08 14:11:58,Gossiping,Military,[新聞] 停火內幕！川普稱「中國促成伊朗談判」,停火 內幕 川普 中國 促成 伊朗 談判 停火 內幕 川普 中國 促成 伊朗 談判 宣稱 美國 完全 勝利 記者 吳美 綜合 美國 伊朗 達成 停火 協議 雙方 宣稱 取得 勝利 預計 巴基斯坦 展開 會談 美國 總統 川普 受訪 ...
1052,2026-04-07 21:15:39,Gossiping,Military,[新聞] 川普警告成真！美軍夜襲伊朗「石油命脈」,川普 警告 成真 美軍 夜襲 伊朗 石油 命脈 記者 閔文昱 川普 警告 成真 美軍 夜襲 伊朗 石油 命脈 哈格 狂傳 爆炸聲 美國 伊朗 緊張 情勢 持續 升溫 外媒 指出 美軍 近日 伊朗 重要 石油 樞紐 哈格 發動 空襲 ...
1061,2026-04-07 17:25:29,Gossiping,Military,[新聞] 力阻川普與伊朗停火！納坦雅胡揚言，就,力阻 川普 伊朗 停火 納坦雅胡 揚言 蘋果 日報 自由時報 參考 版規 下方 核准 名單 直接 官方 允許 風傳媒 記者 記者 名字 名字 外電 至少 法新社 李靖 寫出來 板規 力阻 川普 伊朗 停火 納坦雅胡 揚言 美伊 停戰...
1072,2026-04-07 12:52:29,Gossiping,Military,[新聞] 不看川普臉色！亞洲多國悄悄與伊朗談好,不看 川普 臉色 亞洲 多國 悄悄 伊朗 談好 今日 記者 國際 中心 筱晴 綜合 不看 川普 臉色 亞洲 多國 悄悄 伊朗 談好 荷姆茲海峽 通行證 美國 總統 川普 發出 威脅 美東 週二 點前 伊朗 未能 達成 重啟 荷姆茲海...
1088,2026-04-07 09:54:05,Gossiping,Military,[爆卦] 川普：我們給伊朗人民的槍被庫德族拿走了,川普 伊朗 庫德族 拿走 川普 伊朗 伊朗 本來 送到 伊朗 手中 能夠 擊這些 暴徒 送槍 把槍 扣留 付出 沉重 代價 消息 川普 總統 告訴 福克斯 美國 透過 庫德 人向 伊朗 抗議 運送 槍支 抗議 送去 川普 總統 告訴...
1125,2026-04-06 14:10:47,Gossiping,Military,Re: [問卦] 為什麼伊拉克一打就倒伊朗還能再戰？,伊拉克 一打 伊朗 還能 再戰 中東 海灣 伊拉克 直接 面對 美軍 伊朗 面對 美以 聯軍 伊拉克 一打 伊朗 支持 遜尼派 什葉派 差別 有沒有 懶得 這種 時空 背景 完全 伊拉克 入侵 佔領 科威特 美國 理由 武力 攻擊 ...
1128,2026-04-06 13:29:12,Gossiping,Military,[問卦] 為什麼伊拉克一打就倒伊朗還能再戰？,伊拉克 一打 伊朗 還能 再戰 中東 海灣 伊拉克 直接 面對 美軍 伊朗 面對 美以 聯軍 伊拉克 一打 伊朗 支持 遜尼派 什葉派 差別 有沒有
1164,2026-04-05 14:31:05,Gossiping,Military,[新聞] 專家質疑美國對伊朗Lamerd體育館致命空襲,專家 質疑 美國 伊朗 體育館 致命 空襲 記者 專家 質疑 美國 伊朗 梅爾 體育館 致命 空襲 說法 多名 武器 專家 反駁 美國 說法 美國 聲稱 伊朗 開戰 首日 發生 梅爾 德鎮 一場 致命 空襲 負責 六名 專家 檢視 ...
1180,2026-04-05 12:01:03,Gossiping,Military,Re: [新聞] 伊朗官員放話：還有「大驚喜」等著美國、,伊朗 官員 放話 驚喜 美國 半島 電視台 川普 近期 社群 大秀 炸毀 伊朗 大橋 戰果 威脅 伊朗 送回 石器 時代 言論 該名 具名 伊朗 安全 官員 表現 不屑一顧 美軍 提供 目標 資訊 嚴重 準確 炸毀 橋樑 民生 設施...


## 結論

1. 本 notebook 依照範例 `week7_classification_zh_udn.ipynb` 的分析順序完成，但依 groupwork2 的資料狀況加入了缺值、重複文章與同源後續資料的調整。
2. 清理後的訓練資料共有 2075 篇文章，其中 `Gossiping` 1545 篇、`Military` 530 篇；後續新資料共有 1464 篇文章，時間範圍為 2026-03-21 到 2026-04-10。
3. `CountVectorizer + LogisticRegression` 可以作為基本示範，但改用 `TF-IDF` 後，模型表現更穩定。
4. 在 5-fold CV 的比較下，SVM 與 Random Forest 表現最好；依本次結果，最佳模型會由第 4 節自動選出。
5. 係數分析顯示，`Military` 更偏向軍事情勢與戰況詞彙，`Gossiping` 則更偏向 PTT 問卦與社會討論語氣。
6. 將最佳模型套用到後續新文章後，整體仍維持不錯的分類效果，但 `Military` 的 recall 相對較低，代表部分軍事版文章在語氣上和八卦板文章仍有重疊。
